# TMLR revision experiments — CIs, O2 ImageNet $\Delta$AUC, successful-attack-only ViT

This notebook produces the three quantities the revision needs, using the **same methodology already in the paper**
(stratified bootstrap, $B=2000$, higher score = more adversarial):



In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

RNG_SEED = 0
B = 2000          # bootstrap resamples (paper uses 1000-2000)
ALPHA = 0.05      # 95% CI

def auc(clean, adv):
    "AUROC with higher-score = adversarial (label 1)."
    clean = np.asarray(clean, float); adv = np.asarray(adv, float)
    y = np.concatenate([np.zeros(len(clean)), np.ones(len(adv))])
    s = np.concatenate([clean, adv])
    return roc_auc_score(y, s)

def bootstrap_auc_ci(clean, adv, B=B, seed=RNG_SEED, alpha=ALPHA):
    "Stratified bootstrap CI: resample clean and adv independently (preserves class balance)."
    rng = np.random.default_rng(seed)
    clean = np.asarray(clean, float); adv = np.asarray(adv, float)
    nc, na = len(clean), len(adv)
    stats = np.empty(B)
    for b in range(B):
        cc = clean[rng.integers(0, nc, nc)]
        aa = adv[rng.integers(0, na, na)]
        stats[b] = auc(cc, aa)
    lo, hi = np.quantile(stats, [alpha/2, 1-alpha/2])
    return auc(clean, adv), float(lo), float(hi)

def fmt(point, lo, hi, nd=3):
    return f"{point:.{nd}f} [{lo:.{nd}f}, {hi:.{nd}f}]"

print("utilities ready")


## 1. Load your cached per-sample scores

Return 1-D numpy arrays of detector scores (higher = more adversarial).
Replace the `np.load(...)` lines with your actual cache paths. Keep the return keys.


In [ ]:
# ------------------------------------------------------------------ EDIT PATHS
CACHE = "/path/to/your/score_cache"   # <-- EDIT

def L(name):
    "Helper: load a 1-D score array from the cache."
    import os
    return np.load(os.path.join(CACHE, name))

def load_o2_imagenet():
    """O2 (odds-are-odd) statistic on native ImageNet. Return dict with:
       'clean'   : scores for clean images
       'adv'     : scores for pooled FGSM/PGD/C&W adversarials
       'hardneg' : scores for the unified 3-kind hard negatives (noise sigma=8/255 + JPEG q75 + blur sigma=1)
    """
    return {
        "clean":   L("o2_imagenet_clean.npy"),
        "adv":     L("o2_imagenet_adv_pooled.npy"),
        "hardneg": L("o2_imagenet_hardneg3.npy"),
    }

def load_adaptive():
    """Ensemble detection scores for the 6 Table-11 settings.
       Each entry: (clean_scores, adv_scores) for that setting.
    """
    return {
        "C10 standard C&W (lam=0)":  (L("adap_c10_std_clean.npy"),  L("adap_c10_std_adv.npy")),
        "C10 adaptive (lam=10)":     (L("adap_c10_l10_clean.npy"),  L("adap_c10_l10_adv.npy")),
        "C10 c=0 evasion-only":      (L("adap_c10_c0_clean.npy"),   L("adap_c10_c0_adv.npy")),
        "IN standard C&W (lam=0)":   (L("adap_in_std_clean.npy"),   L("adap_in_std_adv.npy")),
        "IN adaptive (lam=5)":       (L("adap_in_l5_clean.npy"),    L("adap_in_l5_adv.npy")),
        "IN c=0 evasion-only":       (L("adap_in_c0_clean.npy"),    L("adap_in_c0_adv.npy")),
    }

def load_vit():
    """ViT-B/16 detection scores per attack per feature, plus attack-success mask.
       features: HF-Energy, GaussianL1, PredL1, Ensemble.
       For each attack we need clean scores, adv scores, and a boolean success mask
       (True where the attack actually flipped the prediction) aligned to adv scores.
    """
    feats = ["HF-Energy", "GaussianL1", "PredL1", "Ensemble"]
    out = {}
    for atk in ["FGSM", "PGD", "C&W"]:
        succ = L(f"vit_{atk}_success_mask.npy").astype(bool)   # shape (N_adv,)
        per_feat = {}
        for f in feats:
            clean = L(f"vit_{atk}_{f}_clean.npy")
            adv   = L(f"vit_{atk}_{f}_adv.npy")                 # shape (N_adv,)
            per_feat[f] = (clean, adv)
        out[atk] = {"success": succ, "feats": per_feat}
    return out

DATA_READY = False   # <-- set True once paths are correct
print("Set DATA_READY=True after editing paths above, then run the cells below.")


## 2. (C) O2 hard-negative $\Delta$AUC on ImageNet

$\Delta$AUC = AUROC(adv vs. clean $\cup$ hardneg) $-$ AUROC(adv vs. clean).
A near-zero value confirms O2 is *not* fooled by benign perturbations (the paper's claim).
Reported with a paired bootstrap CI on the delta.


In [ ]:
def bootstrap_delta_auc(clean, adv, hardneg, B=B, seed=RNG_SEED, alpha=ALPHA):
    rng = np.random.default_rng(seed)
    clean = np.asarray(clean,float); adv=np.asarray(adv,float); hardneg=np.asarray(hardneg,float)
    def delta(cc, aa, hh):
        before = auc(cc, aa)
        after  = auc(np.concatenate([cc, hh]), aa)
        return after - before
    nc, na, nh = len(clean), len(adv), len(hardneg)
    stats = np.empty(B)
    for b in range(B):
        cc = clean[rng.integers(0,nc,nc)]; aa = adv[rng.integers(0,na,na)]; hh = hardneg[rng.integers(0,nh,nh)]
        stats[b] = delta(cc, aa, hh)
    lo, hi = np.quantile(stats, [alpha/2, 1-alpha/2])
    return delta(clean, adv, hardneg), float(lo), float(hi)

if DATA_READY:
    o2 = load_o2_imagenet()
    before = auc(o2["clean"], o2["adv"])
    after  = auc(np.concatenate([o2["clean"], o2["hardneg"]]), o2["adv"])
    d, dlo, dhi = bootstrap_delta_auc(o2["clean"], o2["adv"], o2["hardneg"])
    print(f"O2 ImageNet clean-only AUROC : {before:.3f}")
    print(f"O2 ImageNet +hard-neg AUROC  : {after:.3f}")
    print(f"O2 ImageNet dAUC             : {d:+.3f}  95% CI [{dlo:+.3f}, {dhi:+.3f}]")
    print()
    print("LaTeX (paste into Sec. 5 interpretation (c), replacing the in-text -0.006):")
    print(rf"$\\Delta$AUC {d:+.3f} on ImageNet ({after:.3f} vs.\\ {before:.3f}; 95\\% CI [{dlo:+.3f},\\,{dhi:+.3f}])")
else:
    print("DATA_READY is False — edit paths in cell 1 first.")


## 3. (B) Bootstrap 95% CIs for Table 11 (adaptive)

Fills the six `[??, ??]` placeholders currently in Table 11.


In [ ]:
if DATA_READY:
    adap = load_adaptive()
    print("Table 11 detection-AUROC with 95% CI:\n")
    latex_cells = {}
    for name, (clean, adv) in adap.items():
        p, lo, hi = bootstrap_auc_ci(clean, adv)
        latex_cells[name] = (p, lo, hi)
        print(f"  {name:28s}  N_adv={len(adv):4d}  {fmt(p,lo,hi)}")
    print("\nPaste-ready det.AUROC cells (replace 0.xxx~[??,~??]):")
    for name,(p,lo,hi) in latex_cells.items():
        print(f"  {name:28s} -> {p:.3f}~[{lo:.3f},~{hi:.3f}]")
else:
    print("DATA_READY is False.")


## 4. (B+D) ViT (Table 10): all-inputs vs. successful-attack-only, both with CIs

- **all** = current paper convention (includes failed attacks).
- **succ** = successful-attack-only (the fix for the FGSM ASR=0.74 confound).

Use the **succ** column as the new Table 10 body; report its CIs (paper's Setup now states this policy).


In [ ]:
def vit_row(clean, adv, mask):
    p_all, lo_all, hi_all = bootstrap_auc_ci(clean, adv)
    adv_s = np.asarray(adv, float)[np.asarray(mask, bool)]
    if len(adv_s) == 0:
        p_s = lo_s = hi_s = float("nan")
    else:
        p_s, lo_s, hi_s = bootstrap_auc_ci(clean, adv_s)
    return (p_all, lo_all, hi_all), (p_s, lo_s, hi_s), len(adv), len(adv_s)

if DATA_READY:
    vit = load_vit()
    feats = ["HF-Energy", "GaussianL1", "PredL1", "Ensemble"]
    print(f"{'attack':6s} {'feature':11s} | {'ALL inputs':22s} | {'SUCCESSFUL-only':22s} | Nadv Nsucc")
    print("-"*95)
    succ_rows = {}   # atk -> {feat: (p,lo,hi)}  (successful-only, for LaTeX)
    for atk in ["FGSM","PGD","C&W"]:
        mask = vit[atk]["success"]
        succ_rows[atk] = {}
        for f in feats:
            clean, adv = vit[atk]["feats"][f]
            allci, sci, na, ns = vit_row(clean, adv, mask)
            succ_rows[atk][f] = sci
            print(f"{atk:6s} {f:11s} | {fmt(*allci):22s} | {fmt(*sci):22s} | {na:4d} {ns:4d}")
    print("\nPaste-ready Table 10 body (successful-attack-only, with CIs):")
    for atk in ["FGSM","PGD","C&W"]:
        cells = " & ".join(f"{p:.3f} [{lo:.3f}, {hi:.3f}]" for (p,lo,hi) in
                           (succ_rows[atk][f] for f in feats))
        print(rf"{atk} & {cells} \\\\")
else:
    print("DATA_READY is False.")


## 5. Checklist after running

1. **(C)** paste the O2 ImageNet $\Delta$AUC string into Sec. 5 interpretation (c); the source has a
   `% NOTE(revision)` marker at that line. Consider adding a small hard-negative table row for O2 so it is tabulated.
2. **(B, Table 11)** replace the six `0.xxx~[??,~??]` cells with the paste-ready values from cell 3.
3. **(B+D, Table 10)** replace the Table 10 body with the successful-attack-only rows from cell 4.
   The Setup paragraph and Table 10 caption already state this policy.
4. Re-`pdflatex` and confirm no `??` placeholders remain: `grep -n '??' main.tex` should be empty.
